# K-ABENA — Niveau 1 : MLP PyTorch
**Coût d'adoption : 2 lignes dans votre boucle d'entraînement existante.**

```python
# AVANT
loss = F.cross_entropy(logits, y)
loss.backward()

# APRÈS (+2 lignes)
losses = F.cross_entropy(logits, y, reduction='none')  # ← 'none'
mask   = kabena_filter_torch(losses, K=0.25)            # ← +1 ligne
losses[mask].mean().backward()                          # ← inchangé
```

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
import numpy as np

from kabena_ml.integrations.torch_utils import kabena_filter_torch, kabena_safe_torch

# Données
X_np, y_np = make_classification(n_samples=1500, n_features=20, random_state=42)
X_np = StandardScaler().fit_transform(X_np)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)

# Modèle — votre MLP habituel, inchangé
model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 2)
)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# Boucle K-ABENA — différence avec SGD standard : 2 lignes
K, N = 0.25, 0.0
for epoch in range(80):
    # ── Avant : loss = F.cross_entropy(model(X), y) ──────────────────────
    losses = F.cross_entropy(model(X), y, reduction='none')  # ← 'none'
    mask   = kabena_filter_torch(losses, K=K, N=N)            # ← +1 ligne
    # ─────────────────────────────────────────────────────────────────────
    L_KA = losses[mask].mean()
    optimizer.zero_grad(); L_KA.backward(); optimizer.step()

    if epoch % 20 == 0:
        m    = mask.sum().item()
        gain = round((1 - m/len(y))*100)
        acc  = (model(X).argmax(1) == y).float().mean().item()
        print(f"Ep {epoch:3d} | loss={L_KA.item():.4f} | m={m}/{len(y)} | "
              f"gain={gain}% | acc={acc:.4f}")

## Avec KabenaTrainer — boucle clé en main

In [ ]:
from kabena_ml.integrations.torch_utils import KabenaTrainer
from kabena_ml import KabenaConfig

model2   = nn.Sequential(nn.Linear(20,64), nn.ReLU(), nn.Linear(64,2))
cfg      = KabenaConfig(K=0.25, N=0.3, task="classification", verbose=True)
trainer  = KabenaTrainer(model2, cfg, lr=0.05, epochs=80)
history  = trainer.fit(X, y)
trainer.plot_stats()
print(f"Accuracy finale : {trainer.evaluate(X, y):.4f}")